# Analise Exploratoria de Dados — Manutencao Preditiva de Motores
## FIAP x Forzy | Challenge IA | Sprint 2

**Dataset:** `motor.db` — 20 motores x 1.500 leituras = **30.000 registros**

**Objetivo:** entender a estrutura dos dados, a distribuicao das classes de falha, o comportamento dos sensores e padroes temporais para guiar a construcao do modelo baseline.

| Secao | Conteudo |
|---|---|
| 1 | Setup e carregamento |
| 2 | Visao geral e qualidade |
| 3 | Distribuicao das classes |
| 4 | Estatisticas dos sensores |
| 5 | Correlacao entre variaveis |
| 6 | Series temporais por falha |
| 7 | Boxplots por classe |

## 1. Setup e carregamento

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

conn = sqlite3.connect('motor.db')
df = pd.read_sql('SELECT * FROM leituras ORDER BY motor_id, timestamp', conn)
tipos = pd.read_sql('SELECT * FROM tipos_falha', conn)
motores = pd.read_sql('SELECT * FROM motores', conn)
conn.close()

df['timestamp'] = pd.to_datetime(df['timestamp'])
nomes_falha = dict(zip(tipos.codigo, tipos.nome))
df['falha_nome'] = df['falha'].map(nomes_falha)
df.head()

## 2. Visao geral e qualidade dos dados

In [ ]:
print('Shape:', df.shape)
print('\nMotores:', df.motor_id.nunique())
print('Leituras por motor:', df.groupby('motor_id').size().unique())
print('\nValores nulos:')
print(df.isnull().sum())
print('\nEstatisticas descritivas:')
df[['temperatura','vibracao','corrente','rotacao']].describe()

## 3. Distribuicao das classes de falha

**Visualizacao 1** — verificar o desbalanceamento (a maioria das leituras e Normal).

In [ ]:
dist = df['falha_nome'].value_counts()
ax = dist.plot(kind='bar', color=['#2E86AB','#F18F01','#C73E1D','#6A994E'])
ax.set_title('Distribuicao das classes de falha (por leitura)')
ax.set_ylabel('Qtd de leituras'); ax.set_xlabel('')
plt.xticks(rotation=20); plt.tight_layout(); plt.savefig('eda_distribuicao.png', dpi=120); plt.show()
print((dist/len(df)*100).round(1))

## 4. Estatisticas dos sensores por classe

**Visualizacao 2** — media de cada sensor por classe de falha.

In [ ]:
medias = df.groupby('falha_nome')[['temperatura','vibracao','corrente','rotacao']].mean()
medias.plot(kind='bar', subplots=True, layout=(2,2), figsize=(12,8), legend=False)
plt.suptitle('Media dos sensores por classe de falha'); plt.tight_layout()
plt.savefig('eda_medias_sensores.png', dpi=120); plt.show()
medias.round(2)

## 5. Correlacao entre variaveis

**Visualizacao 3** — matriz de correlacao dos sensores.

In [ ]:
corr = df[['temperatura','vibracao','corrente','rotacao','falha']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Matriz de correlacao'); plt.tight_layout()
plt.savefig('eda_correlacao.png', dpi=120); plt.show()

## 6. Series temporais por tipo de falha

**Visualizacoes 4-6** — comportamento dos sensores ao longo do tempo para um motor de cada falha (mostra a rampa de deterioracao).

In [ ]:
for cod in [1,2,3]:
    mid = df[df.falha==cod].motor_id.iloc[0]
    sub = df[df.motor_id==mid]
    fig, axes = plt.subplots(2,2, figsize=(12,6))
    for ax,c in zip(axes.ravel(), ['temperatura','vibracao','corrente','rotacao']):
        ax.plot(sub[c].values, color='#C73E1D'); ax.set_title(c)
    fig.suptitle(f'Motor MOT-{mid:03d} — {nomes_falha[cod]}')
    plt.tight_layout(); plt.savefig(f'eda_serie_falha{cod}.png', dpi=120); plt.show()

## 7. Boxplots por classe

**Visualizacao 7** — dispersao de cada sensor por classe, util para ver separabilidade.

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(12,8))
for ax,c in zip(axes.ravel(), ['temperatura','vibracao','corrente','rotacao']):
    sns.boxplot(data=df, x='falha_nome', y=c, ax=ax)
    ax.set_xlabel(''); ax.tick_params(axis='x', rotation=20)
plt.suptitle('Distribuicao dos sensores por classe'); plt.tight_layout()
plt.savefig('eda_boxplots.png', dpi=120); plt.show()

## Conclusoes da EDA

- A base e **fortemente desbalanceada**: ~83% das leituras sao Normais. Isso justifica o uso de `class_weight='balanced'` no modelo.
- Cada falha tem **assinatura propria**: Superaquecimento eleva temperatura/corrente; Desbalanceamento eleva vibracao e derruba RPM; Falha Mecanica combina os tres efeitos.
- As series mostram **rampa progressiva** de deterioracao — features de janela movel (rolling) devem ajudar o modelo a capturar a tendencia.

*FIAP x Forzy | Challenge IA | Sprint 2*